# UÜ Asistan — Adım 3: Cevap üretimi

Test setindeki 84 soruyu uçtan uca çalıştırır (arama → dil modeli): **Qwen3-30B-A3B-Instruct-2507 (4-bit)**.
Her cevap için süre ve token sayısı da kaydedilir. (Qwen3-8B ile karşılaştırma tamamlandı; sonuçlar `eval/sonuclar/` altında.)

**Nasıl çalıştırılır**
1. Çalışma zamanı → Çalışma zamanı türünü değiştir → **L4 GPU**.
2. Hücreleri **yukarıdan aşağı sırayla** çalıştırın. Oturumu yeniden başlatmaya gerek yok.

**Notlar**
- Kod her kurulumda GitHub'dan yeniden indirilir; bu not defterinin eski bir kopyası açık olsa bile en güncel kod çalışır.
- Modeller ayrı süreçlerde çalışır (`!python ...`); not defteri PyTorch'u kendisi yüklemez.
- Sonuçlar `/content/sonuclar/` klasörüne yazılır; kurulum yeniden çalıştırılsa da silinmez.
- Süreler: kurulum ~5 dk, deneme ~5 dk (model indirme dahil), tam koşu ~10–15 dk.

## 1. Kurulum

In [ ]:
import os, subprocess

# Çalışma klasörü silinmiş bir klasör olabilir (önceki oturumlar); her şeyden önce sabit bir yere geç.
os.chdir("/content")

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "GPU yok: Çalışma zamanı → Çalışma zamanı türünü değiştir → L4 GPU"
print("GPU:", gpu)

# Her seferinde temiz kopya: eski kod ya da önceki koşulardan kalan dosyalar karışmasın.
!rm -rf /content/uu-asistan
!git clone -q --depth 1 https://github.com/Cnrygt11/uu-asistan.git /content/uu-asistan
assert os.path.exists("/content/uu-asistan/uuasistan/uretim.py"), "Kod indirilemedi (git clone çıktısına bakın)"
!git -C /content/uu-asistan log --oneline -1

!pip install -q -r /content/uu-asistan/requirements-arama.txt

# Kurulumu ayrı bir süreçte doğrula (bu not defterinin süreci PyTorch yüklemez).
dogrulama = subprocess.run(
    ["python", "-c", "import torch, vllm; print('torch', torch.__version__, '| vLLM', vllm.__version__, '| CUDA', torch.cuda.is_available())"],
    capture_output=True, text=True)
print(dogrulama.stdout or dogrulama.stderr[-2000:])
assert dogrulama.returncode == 0 and "CUDA True" in dogrulama.stdout, "vLLM/PyTorch kurulumu başarısız"

os.makedirs("/content/sonuclar", exist_ok=True)
print("Kurulum tamam.")

## 2. Deneme (3 soru)

Tam koşudan önce modelin yüklendiğini ve cevap verdiğini görmek için. Hata olursa bütün çıktıyı paylaşın.

In [ ]:
!cd /content/uu-asistan && python -m uuasistan.uretim --model qwen3-30b-a3b --sinir 3

## 3. Tam koşu (84 soru)

In [ ]:
!cd /content/uu-asistan && python -m uuasistan.uretim --model qwen3-30b-a3b --cikti /content/sonuclar/adim3_qwen3-30b-a3b_v2.jsonl

## 4. Sonuçları indir

İnen dosyaları depodaki `eval/sonuclar/` klasörüne koyun.

In [ ]:
import os
from google.colab import files

for ad in ("adim3_qwen3-30b-a3b_v2.jsonl",):
    yol = f"/content/sonuclar/{ad}"
    if os.path.exists(yol):
        files.download(yol)
    else:
        print("Bulunamadı (koşu tamamlanmamış olabilir):", yol)